In [11]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from xgboost import XGBClassifier


In [12]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("muhammadkhubaibahmad/confidence-detection-dataset")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/datasets/muhammadkhubaibahmad/confidence-detection-dataset


In [13]:
import pandas as pd
import os

csv_path = "/kaggle/input/datasets/muhammadkhubaibahmad/confidence-detection-dataset/confidence_features.csv"

df = pd.read_csv(csv_path)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Shape: (5949, 19)

Columns:
['eye_shoulder_y_ratio', 'shoulder_y_diff', 'wrist_distance_x', 'wrist_shoulder_ratio', 'nose_eye_center_offset_x', 'shoulder_span', 'hip_shoulder_y_diff', 'body_lean_x', 'shoulder_center_x', 'hip_center_x', 'spine_angle', 'eye_distance', 'head_tilt_angle', 'eye_distance_ratio', 'shoulder_slope', 'head_direction', 'arm_position', 'posture', 'confidence_label']


In [15]:
TARGET = "confidence_label"

X = df.drop(columns=[TARGET])
y = df[TARGET]



In [16]:
categorical_cols = [
    "head_direction",
    "arm_position",
    "posture"
]

encoders = {}

for col in categorical_cols:
    encoder = LabelEncoder()

    X[col] = encoder.fit_transform(
        X[col].astype(str)
    )

    encoders[col] = encoder


# Encode target if necessary
target_encoder = LabelEncoder()

y = target_encoder.fit_transform(
    y.astype(str)
)

print("\nClasses:")
print(target_encoder.classes_)


Classes:
['Confident' 'Low' 'Neutral']


In [17]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.20,random_state=42,stratify=y)
print("\nTraining:", X_train.shape)
print("Testing :", X_test.shape)


Training: (4759, 18)
Testing : (1190, 18)


In [19]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=500,
    max_depth=5,
    learning_rate=0.03,
    subsample=0.85,
    colsample_bytree=0.85,
    min_child_weight=2,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,

    objective="multi:softprob",
    num_class=3,
    eval_metric="mlogloss",

    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

model.fit(
    X_train,
    y_train
)
print("Training completed!")

Training completed!


In [22]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

y_pred = model.predict(X_test)

accuracy = accuracy_score(
    y_test,
    y_pred
)
print(accuracy)

0.9857142857142858


In [23]:
print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=target_encoder.classes_
    )
)


Classification Report:
              precision    recall  f1-score   support

   Confident       0.98      1.00      0.99       627
         Low       1.00      0.99      0.99       231
     Neutral       0.99      0.96      0.97       332

    accuracy                           0.99      1190
   macro avg       0.99      0.98      0.99      1190
weighted avg       0.99      0.99      0.99      1190



In [24]:
print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test,
        y_pred
    )
)


Confusion Matrix:
[[625   0   2]
 [  0 229   2]
 [ 12   1 319]]


In [25]:
import joblib

joblib.dump(
    model,
    "/kaggle/working/confidence_xgboost_model.pkl"
)

print("Model saved successfully!")

Model saved successfully!
